# PDF reporting for HDAB id project

This script uses Python and the ReportLab toolkit https://www.reportlab.com/, the PyPDF module https://pypdf.readthedocs.io/en/stable/ to merge generated PDF files, and some basic modules. 


To do:
- Link to database when available based on view with left join morphospeciescodes
- Add argpase to run as a script
    - include option to combine multiple PDFs
    - include wishlist option to supply a csv with list of morphospeciescodes to generate report for
    - include argument to set directory with images

## Import dependencies, fonts, and set working directories

In [70]:
import os
import pandas as pd
import reportlab
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from pypdf import PdfWriter
from datetime import date
from pathlib import Path

font_dir = "./fonts/" 
arial_reg = os.path.join(font_dir, "Arial.ttf")
arial_bold = os.path.join(font_dir, "Arial Bold.ttf")
arial_italic = os.path.join(font_dir, "Arial Italic.ttf")
arial_bolditalic = os.path.join(font_dir, "Arial Bold Italic.ttf")

pdfmetrics.registerFont(TTFont('Arial', arial_reg))
pdfmetrics.registerFont(TTFont('Arial-Bold', arial_bold))
pdfmetrics.registerFont(TTFont('Arial-Italic', arial_italic))
pdfmetrics.registerFont(TTFont('Arial-BoldItalic', arial_bolditalic))

img_dir = "./test_imgs/"
individualreports_dir = "Individual_reports"
os.makedirs(individualreports_dir, exist_ok=True) # creates output folder if it does not exist
combinedreports_dir = "Combined_reports"
os.makedirs(combinedreports_dir, exist_ok=True)

In [71]:
print(f"Using ReportLab version: {reportlab.Version}")

Using ReportLab version: 3.5.67


## Function to create a ReportLab PDFs with info from dataframe and add images

In [72]:
def pdfreporter(df, individualreports_dir, img_dir):
    for row in df.sort_values(by=['hdoareference', 'morphospeciescode']).itertuples():
        morphospeciescode = row.morphospeciescode
        print("Creating PDF report for: " + (morphospeciescode))
        # Create canvas with morphospeciescode filename in subfolder Reports
        filename = str(morphospeciescode) + ".pdf"
        full_path = os.path.join(individualreports_dir, filename)
        c = canvas.Canvas(full_path, pagesize=letter) # Letter size; width: 612, height: 792 points
        page_width, page_height = letter
        # Add header image
        UHIMlogo = "./logos/header1.png"
        img_width = 530
        img_height = 90
        x_coord = (page_width - img_width) / 2 # to center image on page
        c.drawImage(UHIMlogo, x_coord, 680, width=img_width, height=img_height, preserveAspectRatio=True)
        # Add title and date
        x_center = page_width / 2 # to center text on page
        c.setFont("Arial-Bold", 14) # font and size
        c.drawCentredString(x_center, 655, "University of Hawaiʻi Insect Museum")
        c.drawCentredString(x_center, 635, "Arthropod Identification Report")
        c.setFont("Arial", 10)
        today = date.today()
        c.drawCentredString(x_center, 620, ("Report generated: " + today.strftime("%B %d, %Y")))
        
        # Add information from dataframe
        c.setFont("Arial", 11) # keep 12 y value spacing between text with font size 11
        
        c.drawString(48, 586, "Project: Hawaiʻi Department of Agriculture and Biosecurity RFP-25-06-PI. Fiscal year 2025.")
    
        c.drawString(48, 567, "HDAB reference: " + str(row.hdoareference))
        c.drawString(48, 555, "Date collected: " + str(row.datecollected))
        c.drawString(48, 543, "Origin: " + str(row.origin))
        c.drawString(48, 531, "Intercepted host: " + str(row.host))
        c.drawString(48, 519, "Number of species in sample: " + str(row.speciescount))
    
        c.drawString(48, 495, "UHIM identification reference: " + str(morphospeciescode))
        c.drawString(48, 483, "Number of specimens of this species: " + str(row.specimencount))
    
        c.drawString(48, 459, "Integrative identification:")
        if row.genusorlower == 1:
            c.setFont("Arial-Italic", 11)
        else: c.setFont("Arial", 11)
        c.drawString(169, 459, str(row.finalid))
        c.setFont("Arial", 11)
        c.drawString(48, 447, "Common name: " + str(row.finalidcommonname))
        c.drawString(48, 435, "Order: " + str(row.finalidorder))
        c.drawString(48, 423, "Family: " + str(row.finalidfamily))
    
        c.drawString(48, 399, "Identification notes: Camiel thinks this might be a spider")
        c.drawString(48, 387, "Distribution notes: Only known from The Netherlands")
        
        # Add photo of morphotype
        # get a list of photos in the img_dir
        jpg_filenames = [
            f for f in os.listdir(img_dir) 
            if f.lower().endswith('.jpg') and os.path.isfile(os.path.join(img_dir, f))
        ]
        # Select image file names containing "morphospeciescode"
        morphospeciesimages = [item for item in jpg_filenames if morphospeciescode in item]
        if len(morphospeciesimages) == 0:
            print("   No images found, continuing without image")
        else:
            # Select the first photo in the list to print
            speciesimage = os.path.join(img_dir, morphospeciesimages[0])
            # Print on page
            img_width = 500
            img_height = 400
            x_coord = (page_width - img_width) / 2
            c.drawImage(speciesimage, x_coord, 10, width=img_width, height=img_height, preserveAspectRatio=True)
            # Add image filename underneath
            imgfilename = os.path.basename(speciesimage)
            c.setFont("Arial", 10) # font and size
            c.drawCentredString(x_center, 60, imgfilename)
            
        c.showPage() # finish page
        c.save() # construct and save file to .pdf
        print("   Successfully created report " + str(filename))

## Function to merge individual PDFs into one in separate folder

In [91]:
def combinepdf(individualreports_dir):
    pdf_with_folder = [
        os.path.join(individualreports_dir, f) 
        for f in os.listdir(individualreports_dir) 
        if f.lower().endswith('.pdf') and os.path.isfile(os.path.join(individualreports_dir, f))
    ]
    writer = PdfWriter()
    for pdf in pdf_with_folder:
        writer.append(pdf)
    today = date.today()
    #today = date.now().strftime("%y%m%d")
#    today.strftime("%B %d, %Y")
    writer.write("./Combined_reports/" + today.strftime("%y%m%d") + "_combined_report.pdf")
    writer.close()
    print(f"Successfully merged {len(pdf_files)} files")

## Import the data to be reported from a csv file into a pandas dataframe

This will be changed to a postgreSQL pull once the database is set up.

In [92]:
df = pd.read_csv('./test_input_tables/testIDs.csv')
print(df.head())

       hdoareference datecollected        host      origin  morphospeciescode  \
0  Interception 1103        1/1/26  Strawberry  California  HDOA260619_113_M3   
1   Interception 111        3/4/22        Mint      Mexico  HDOA250917_034_M2   
2   Interception 111   May 12 2010        Mint      Mexico  HDOA250917_034_M1   

  finalidfamily finalidorder        finalid  genusorlower finalidcommonname  \
0  Dontknowidae  Lepidoptera   Lorem Ipsum1         False               Bug   
1      Maybidae   Coleoptera  Delia platura          True            Spider   
2      Testidae    Hemiptera   Lorem Ipsum3         False              Bug3   

              distributionnotes  hostnotes  speciescount  specimencount  
0                    New for HI        NaN             4             40  
1  Cosmopolitan, all HI islands        NaN             3             10  
2  Cosmopolitan, all HI islands        NaN             6              1  


## Run functions

In [93]:
# generate a PDF report for each morphospecies
try:
    pdfreporter(df, individualreports_dir, img_dir)
except Exception as e:
    print(f"An error occurred: {e}")

# combine reports into one PDF (keeps separate files too)
try:
    combinepdf(individualreports_dir)
except Exception as e:
    print(f"An error occurred: {e}")

Creating PDF report for: HDOA260619_113_M3
   Successfully created report HDOA260619_113_M3.pdf
Creating PDF report for: HDOA250917_034_M1
   No images found, continuing without image
   Successfully created report HDOA250917_034_M1.pdf
Creating PDF report for: HDOA250917_034_M2
   No images found, continuing without image
   Successfully created report HDOA250917_034_M2.pdf
Successfully merged 3 files


## PostgreSQL connection

In [ ]:
import psycopg2

connectstringfile = str('.connectstring_' + database)
if os.path.exists('./.connectstring_' + database) == False:
    sys.exit("Missing .connectstring file for database " + database + " : stopping")
connectstring = linecache.getline(filename=connectstringfile, lineno=1).rstrip('\n')
conn = psycopg2.connect(connectstring)
if conn.closed == 0:
    print("Successfully connected to psql database")
else:
    sys.exit("Could not connect to psql database: stopping") 
conn = None

## Pull data from postgreSQL database

In [ ]:
conn = psycopg2.connect(connectstring)
sql = "SELECT markerset FROM dna_markersets;"
df_markersetoverview = pd.read_sql_query(sql, conn)
conn = None